<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/15A1_NeuroFHIR_Review_WISH_Professional_UI_Facelift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-Review — Notebook 15A1
## Professional UI Facelift — **without changing the study**

This notebook upgrades the visual design of the frozen **12-case NeuroFHIR-Review** participant application while preserving the experimental protocol exactly.

### Design goal
Make `/wish-review/` feel like the same professional NeuroFHIR product family as the original AMIA application:

- polished clinical-research header;
- restrained navy / teal visual language;
- stronger spacing and typography;
- clear cards and section hierarchy;
- better case-progress visibility;
- professional form controls and action buttons;
- neutral AI styling so the interface does **not visually privilege the AI recommendation**;
- responsive desktop / tablet / mobile layout;
- accessible focus states and contrast;
- research-prototype footer.

### Scientific constraints
This notebook **does not**:
- reorder any study stage;
- change Evidence-First / AI-First assignment;
- alter the 12 cases;
- alter AI recommendations, uncertainty, QC, provenance, or reference content;
- change form values, button labels, event names, timestamps, exports, or participant IDs;
- expose researcher-only information;
- add external fonts, icon libraries, analytics, or network dependencies.

The facelift is injected around the existing frozen app. A round-trip integrity test removes the injected UI blocks and requires the remaining HTML to be **byte-for-byte identical** to the original.

## Correct project order

`13 → 14 → 14B → 14C → 15 → 15A1 UI facelift → 15A redeploy → 15B live QA → Step 9 real reviewers → 16 analysis`

**P001/P002 remain dry-run IDs. Real participants start with P003.**

In [1]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Cell 2 — Configuration
from pathlib import Path

DRIVE_REPO_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
WISH_ROOT = DRIVE_REPO_ROOT / "wish_extension"

FROZEN_APP = WISH_ROOT / "final_wish_pilot" / "participant_app"
INDEX = FROZEN_APP / "index.html"

BACKUP_DIR = WISH_ROOT / "final_wish_pilot" / "participant_app_pre_ui_facelift"
CANDIDATE_DIR = WISH_ROOT / "final_wish_pilot" / "participant_app_ui_candidate"

# The notebook makes a backup and promotes only after all integrity gates pass.
PROMOTE_TO_FROZEN_APP = True

assert INDEX.exists(), f"Frozen participant app not found: {INDEX}"

print("Frozen app:", FROZEN_APP)
print("Index:", INDEX)
print("Promote after gates:", PROMOTE_TO_FROZEN_APP)

Frozen app: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app
Index: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app/index.html
Promote after gates: True


In [3]:
# Cell 3 — Read original and create a one-time backup
import shutil, hashlib, json, re, os, datetime

original_html = INDEX.read_text(encoding="utf-8")

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def sha256_file(p: Path) -> str:
    return sha256_bytes(p.read_bytes())

original_index_hash = sha256_file(INDEX)

# Stable pre-facelift backup. Never overwrite an existing backup.
if not BACKUP_DIR.exists():
    shutil.copytree(FROZEN_APP, BACKUP_DIR)
    print("✅ Created pre-facelift backup:", BACKUP_DIR)
else:
    print("ℹ️ Pre-facelift backup already exists:", BACKUP_DIR)

backup_index = BACKUP_DIR / "index.html"
assert backup_index.exists()
assert sha256_file(backup_index) == original_index_hash, (
    "Existing backup does not match the currently frozen index.html. "
    "STOP and inspect before continuing."
)

print("Original index SHA-256:", original_index_hash)
print("Original HTML bytes:", len(original_html.encode("utf-8")))

✅ Created pre-facelift backup: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app_pre_ui_facelift
Original index SHA-256: 40ed5f33e47ddea1716f4c63626c158156fdd613d177a489bbfa825e72264e29
Original HTML bytes: 15181


In [4]:
# Cell 4 — Snapshot study-critical structure before styling
#
# We deliberately record the existing scripts and common study markers before
# any visual work. These must remain present afterward.

SCRIPT_RE = re.compile(r"<script\b[^>]*>.*?</script>", re.I | re.S)
scripts_before = SCRIPT_RE.findall(original_html)
script_hashes_before = [sha256_bytes(x.encode("utf-8")) for x in scripts_before]

critical_markers = [
    "NeuroFHIR-Review",
    "initial_judgment",
    "initial_confidence",
    "provenance_opened",
    "final_action",
    "final_confidence",
    "reason_code",
    "rationale",
    "initial_submitted_utc",
    "final_submitted_utc",
]

present_markers = [m for m in critical_markers if m.lower() in original_html.lower()]

print("Existing <script> blocks:", len(scripts_before))
print("Detected study-critical markers:", present_markers)

assert len(scripts_before) >= 1, "No script block detected in participant app."
assert "neurofhir-review" in original_html.lower(), (
    "This does not look like the NeuroFHIR-Review participant app."
)

print("✅ Pre-facelift study snapshot: PASS")

Existing <script> blocks: 1
Detected study-critical markers: ['NeuroFHIR-Review', 'initial_judgment', 'initial_confidence', 'provenance_opened', 'final_action', 'final_confidence', 'reason_code', 'rationale', 'initial_submitted_utc', 'final_submitted_utc']
✅ Pre-facelift study snapshot: PASS


In [5]:
# Cell 5 — Build the professional visual layer
#
# IMPORTANT:
# - no existing element is reordered;
# - no existing input/button value is changed;
# - no existing script is edited;
# - styling is neutral with respect to AI vs human evidence.

FACELIFT_STYLE = r"""
<!-- NF_UI_FACELIFT_STYLE_BEGIN -->
<style id="nf-wish-professional-ui">
:root {
  --nf-navy-950: #071b2c;
  --nf-navy-900: #0b253b;
  --nf-navy-800: #123b56;
  --nf-teal-700: #0f6d78;
  --nf-teal-600: #16818d;
  --nf-teal-100: #e8f5f6;
  --nf-blue-100: #edf5fb;
  --nf-slate-900: #182431;
  --nf-slate-700: #475868;
  --nf-slate-600: #607181;
  --nf-slate-500: #748492;
  --nf-slate-300: #ccd6de;
  --nf-slate-200: #dde5eb;
  --nf-slate-100: #edf2f6;
  --nf-slate-050: #f6f8fb;
  --nf-white: #ffffff;
  --nf-warning-bg: #fff8e8;
  --nf-warning-border: #e7c96d;
  --nf-danger-bg: #fff2f1;
  --nf-danger-border: #dfaaa5;
  --nf-success-bg: #eef8f3;
  --nf-success-border: #afd9c4;
  --nf-shadow-sm: 0 1px 2px rgba(7,27,44,.05), 0 1px 5px rgba(7,27,44,.04);
  --nf-shadow-md: 0 10px 30px rgba(7,27,44,.08), 0 2px 8px rgba(7,27,44,.05);
  --nf-radius-sm: 8px;
  --nf-radius-md: 12px;
  --nf-radius-lg: 18px;
  --nf-content: 1180px;
}

* { box-sizing: border-box; }

html {
  background: var(--nf-slate-050);
  scroll-behavior: smooth;
}

body {
  margin: 0 !important;
  min-height: 100vh;
  color: var(--nf-slate-900);
  background:
    radial-gradient(circle at 10% -10%, rgba(22,129,141,.08), transparent 28rem),
    linear-gradient(180deg, #f8fafc 0%, #f4f7fa 100%) !important;
  font-family:
    Inter, ui-sans-serif, -apple-system, BlinkMacSystemFont, "Segoe UI",
    Roboto, Helvetica, Arial, sans-serif !important;
  font-size: 16px;
  line-height: 1.55;
  -webkit-font-smoothing: antialiased;
  text-rendering: optimizeLegibility;
}

/* ---------- Product header ---------- */
#nf-study-shell {
  position: sticky;
  top: 0;
  z-index: 9999;
  width: 100%;
  background: linear-gradient(120deg, var(--nf-navy-950), var(--nf-navy-800));
  color: var(--nf-white);
  box-shadow: 0 8px 24px rgba(7,27,44,.14);
}

.nf-shell-inner {
  max-width: var(--nf-content);
  margin: 0 auto;
  padding: 18px 28px 15px;
}

.nf-shell-top {
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 24px;
}

.nf-brand {
  display: flex;
  align-items: center;
  gap: 13px;
  min-width: 0;
}

.nf-mark {
  width: 42px;
  height: 42px;
  flex: 0 0 42px;
  display: grid;
  place-items: center;
  border-radius: 12px;
  background: rgba(255,255,255,.10);
  border: 1px solid rgba(255,255,255,.18);
  font-weight: 800;
  letter-spacing: -.04em;
  box-shadow: inset 0 1px 0 rgba(255,255,255,.10);
}

.nf-brand-copy { min-width: 0; }

.nf-brand-title {
  font-size: 18px;
  font-weight: 750;
  line-height: 1.15;
  letter-spacing: -.015em;
}

.nf-brand-subtitle {
  margin-top: 3px;
  color: rgba(255,255,255,.72);
  font-size: 12.5px;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
}

.nf-study-badge {
  flex: 0 0 auto;
  padding: 7px 10px;
  border-radius: 999px;
  color: rgba(255,255,255,.91);
  background: rgba(255,255,255,.08);
  border: 1px solid rgba(255,255,255,.16);
  font-size: 12px;
  font-weight: 650;
  letter-spacing: .01em;
}

.nf-progress-wrap {
  display: flex;
  align-items: center;
  gap: 12px;
  margin-top: 14px;
}

.nf-progress-track {
  position: relative;
  flex: 1;
  height: 5px;
  overflow: hidden;
  border-radius: 999px;
  background: rgba(255,255,255,.14);
}

#nf-progress-fill {
  width: 0%;
  height: 100%;
  border-radius: inherit;
  background: #7fd8dd;
  transition: width .25s ease;
}

#nf-progress-label {
  min-width: 88px;
  text-align: right;
  color: rgba(255,255,255,.72);
  font-size: 12px;
  font-variant-numeric: tabular-nums;
}

/* ---------- Main app layout ---------- */
body > :not(#nf-study-shell):not(#nf-study-footer) {
  max-width: var(--nf-content);
  margin-left: auto !important;
  margin-right: auto !important;
}

main,
[role="main"],
.nf-main {
  width: min(var(--nf-content), calc(100% - 40px));
  margin: 28px auto 44px !important;
}

h1, h2, h3, h4 {
  color: var(--nf-navy-900);
  line-height: 1.25;
  letter-spacing: -.02em;
}

h1 {
  margin-top: 0;
  font-size: clamp(25px, 3vw, 34px);
  font-weight: 760;
}

h2 {
  font-size: clamp(20px, 2.2vw, 25px);
  font-weight: 730;
}

h3 {
  font-size: 17px;
  font-weight: 720;
}

p { color: var(--nf-slate-700); }

/* ---------- Cards / panels ---------- */
.nf-section-card,
section,
article,
fieldset,
.card,
.panel {
  border-radius: var(--nf-radius-lg);
}

.nf-section-card {
  margin: 18px 0 !important;
  padding: 22px 24px !important;
  background: rgba(255,255,255,.96) !important;
  border: 1px solid var(--nf-slate-200) !important;
  box-shadow: var(--nf-shadow-sm) !important;
}

.nf-section-card > :first-child { margin-top: 0 !important; }
.nf-section-card > :last-child { margin-bottom: 0 !important; }

.nf-card-eyebrow {
  display: inline-flex;
  align-items: center;
  gap: 7px;
  margin-bottom: 10px;
  padding: 5px 8px;
  border-radius: 7px;
  color: var(--nf-teal-700);
  background: var(--nf-teal-100);
  font-size: 11px;
  font-weight: 750;
  letter-spacing: .055em;
  text-transform: uppercase;
}

/* The AI card is intentionally not visually louder than the evidence card. */
.nf-ai-card,
.nf-evidence-card,
.nf-passport-card,
.nf-judgment-card,
.nf-final-card {
  border-left: 4px solid var(--nf-slate-300) !important;
}

.nf-ai-card { border-left-color: #8aa6b8 !important; }
.nf-evidence-card { border-left-color: var(--nf-teal-600) !important; }
.nf-passport-card { border-left-color: #6f8ca3 !important; }
.nf-judgment-card { border-left-color: #8e9fb0 !important; }
.nf-final-card { border-left-color: var(--nf-navy-800) !important; }

/* ---------- Clinical metadata ---------- */
table {
  width: 100%;
  border-collapse: separate;
  border-spacing: 0;
  overflow: hidden;
  background: var(--nf-white);
  border: 1px solid var(--nf-slate-200);
  border-radius: var(--nf-radius-md);
}

th, td {
  padding: 11px 13px;
  text-align: left;
  border-bottom: 1px solid var(--nf-slate-100);
  vertical-align: top;
}

th {
  color: var(--nf-slate-700);
  background: #f7f9fb;
  font-size: 12px;
  font-weight: 720;
}

tr:last-child td { border-bottom: 0; }

pre, code {
  font-family: "SFMono-Regular", Consolas, "Liberation Mono", Menlo, monospace;
}

pre {
  overflow: auto;
  padding: 16px;
  border: 1px solid var(--nf-slate-200);
  border-radius: var(--nf-radius-md);
  color: #243746;
  background: #f7f9fb;
  font-size: 12.5px;
  line-height: 1.5;
}

/* ---------- Imaging / evidence ---------- */
img, canvas, svg {
  max-width: 100%;
}

img {
  border-radius: 12px;
}

.nf-section-card img,
.nf-section-card canvas {
  box-shadow: 0 0 0 1px rgba(7,27,44,.08), var(--nf-shadow-sm);
}

/* ---------- Forms ---------- */
label {
  color: var(--nf-slate-900);
  font-weight: 610;
}

input[type="text"],
input[type="number"],
select,
textarea {
  width: 100%;
  padding: 11px 12px;
  border: 1px solid #bcc9d3;
  border-radius: 10px;
  color: var(--nf-slate-900);
  background: var(--nf-white);
  font: inherit;
  outline: none;
  transition: border-color .15s ease, box-shadow .15s ease;
}

textarea {
  min-height: 112px;
  resize: vertical;
}

input[type="text"]:focus,
input[type="number"]:focus,
select:focus,
textarea:focus {
  border-color: var(--nf-teal-600);
  box-shadow: 0 0 0 3px rgba(22,129,141,.12);
}

input[type="radio"],
input[type="checkbox"] {
  accent-color: var(--nf-teal-700);
  width: 17px;
  height: 17px;
}

fieldset {
  margin: 16px 0;
  padding: 17px 18px;
  border: 1px solid var(--nf-slate-200);
  background: #fbfcfd;
}

legend {
  padding: 0 6px;
  color: var(--nf-navy-900);
  font-weight: 700;
}

/* ---------- Buttons ---------- */
button,
input[type="button"],
input[type="submit"] {
  min-height: 42px;
  padding: 10px 16px;
  border: 1px solid transparent;
  border-radius: 10px;
  color: var(--nf-white);
  background: var(--nf-navy-800);
  font: inherit;
  font-weight: 680;
  letter-spacing: -.005em;
  cursor: pointer;
  box-shadow: 0 1px 2px rgba(7,27,44,.08);
  transition:
    transform .08s ease,
    box-shadow .15s ease,
    background-color .15s ease,
    border-color .15s ease;
}

button:hover,
input[type="button"]:hover,
input[type="submit"]:hover {
  background: var(--nf-navy-900);
  box-shadow: 0 4px 12px rgba(7,27,44,.12);
}

button:active,
input[type="button"]:active,
input[type="submit"]:active {
  transform: translateY(1px);
}

button:focus-visible,
input:focus-visible,
select:focus-visible,
textarea:focus-visible,
a:focus-visible {
  outline: 3px solid rgba(22,129,141,.28);
  outline-offset: 2px;
}

button:disabled,
input[type="submit"]:disabled {
  cursor: not-allowed;
  opacity: .48;
  box-shadow: none;
}

.nf-secondary-button {
  color: var(--nf-navy-800) !important;
  background: var(--nf-white) !important;
  border-color: #b8c6d0 !important;
}

.nf-secondary-button:hover {
  background: var(--nf-slate-050) !important;
  border-color: #90a5b5 !important;
}

/* ---------- Status / callout styling ---------- */
[role="alert"],
.alert,
.warning,
.error,
.success,
.notice {
  padding: 13px 15px;
  border-radius: 10px;
}

.warning { background: var(--nf-warning-bg); border: 1px solid var(--nf-warning-border); }
.error { background: var(--nf-danger-bg); border: 1px solid var(--nf-danger-border); }
.success { background: var(--nf-success-bg); border: 1px solid var(--nf-success-border); }

/* ---------- Research footer ---------- */
#nf-study-footer {
  max-width: none !important;
  margin: 50px 0 0 !important;
  padding: 0 24px 26px;
}

.nf-footer-inner {
  max-width: var(--nf-content);
  margin: 0 auto;
  padding-top: 18px;
  border-top: 1px solid var(--nf-slate-200);
  display: flex;
  justify-content: space-between;
  gap: 18px;
  color: var(--nf-slate-500);
  font-size: 11.5px;
}

.nf-footer-inner strong {
  color: var(--nf-slate-700);
}

/* ---------- Responsive ---------- */
@media (max-width: 760px) {
  .nf-shell-inner { padding: 14px 16px 12px; }
  .nf-shell-top { align-items: flex-start; }
  .nf-brand-subtitle { white-space: normal; }
  .nf-study-badge { display: none; }

  body > :not(#nf-study-shell):not(#nf-study-footer) {
    max-width: calc(100% - 22px);
  }

  main,
  [role="main"],
  .nf-main {
    width: calc(100% - 22px);
    margin-top: 16px !important;
  }

  .nf-section-card {
    padding: 17px 15px !important;
    border-radius: 14px;
  }

  .nf-footer-inner { flex-direction: column; gap: 6px; }
}

@media (max-width: 480px) {
  .nf-mark { width: 36px; height: 36px; flex-basis: 36px; }
  .nf-brand-title { font-size: 16px; }
  .nf-brand-subtitle { font-size: 11.5px; }
  #nf-progress-label { min-width: 70px; }
  button,
  input[type="button"],
  input[type="submit"] { width: 100%; }
}

@media (prefers-reduced-motion: reduce) {
  *, *::before, *::after {
    scroll-behavior: auto !important;
    transition: none !important;
    animation: none !important;
  }
}

@media print {
  #nf-study-shell,
  #nf-study-footer { display: none !important; }
  body { background: #fff !important; }
  .nf-section-card { box-shadow: none !important; }
}
</style>
<!-- NF_UI_FACELIFT_STYLE_END -->
"""

FACELIFT_HEADER = r"""
<!-- NF_UI_FACELIFT_HEADER_BEGIN -->
<header id="nf-study-shell" aria-label="NeuroFHIR-Review study header">
  <div class="nf-shell-inner">
    <div class="nf-shell-top">
      <div class="nf-brand">
        <div class="nf-mark" aria-hidden="true">NF</div>
        <div class="nf-brand-copy">
          <div class="nf-brand-title">NeuroFHIR-Review</div>
          <div class="nf-brand-subtitle">Longitudinal neuroimaging evidence · uncertainty · provenance · accountable review</div>
        </div>
      </div>
      <div class="nf-study-badge">Research prototype · 12-case review</div>
    </div>
    <div class="nf-progress-wrap" aria-label="Case progress">
      <div class="nf-progress-track" aria-hidden="true">
        <div id="nf-progress-fill"></div>
      </div>
      <div id="nf-progress-label">Review session</div>
    </div>
  </div>
</header>
<!-- NF_UI_FACELIFT_HEADER_END -->
"""

FACELIFT_FOOTER = r"""
<!-- NF_UI_FACELIFT_FOOTER_BEGIN -->
<footer id="nf-study-footer">
  <div class="nf-footer-inner">
    <div><strong>NeuroFHIR-Review</strong> · formative human–AI research workflow</div>
    <div>Research prototype · not for patient care or diagnostic use</div>
  </div>
</footer>
<!-- NF_UI_FACELIFT_FOOTER_END -->
"""

FACELIFT_SCRIPT = r"""
<!-- NF_UI_FACELIFT_SCRIPT_BEGIN -->
<script id="nf-wish-professional-ui-script">
(function () {
  "use strict";

  function textOf(el) {
    return (el && el.textContent ? el.textContent : "").replace(/\s+/g, " ").trim();
  }

  function nearestCardCandidate(el) {
    if (!el) return null;
    return el.closest(
      "section, article, fieldset, form, .card, .panel, .section, .content, .step, div"
    );
  }

  function decorateSections() {
    var headings = document.querySelectorAll("h1,h2,h3,h4,[role='heading']");
    headings.forEach(function (h) {
      var t = textOf(h).toLowerCase();
      var card = nearestCardCandidate(h);

      if (!card || card.id === "nf-study-shell" || card.id === "nf-study-footer") return;

      var matched = false;

      if (t.includes("case brief") || t === "case") {
        card.classList.add("nf-section-card", "nf-evidence-card");
        matched = true;
      } else if (t.includes("evidence passport") || t.includes("provenance")) {
        card.classList.add("nf-section-card", "nf-passport-card");
        matched = true;
      } else if (t.includes("initial judgment")) {
        card.classList.add("nf-section-card", "nf-judgment-card");
        matched = true;
      } else if (
        t.includes("ai recommendation") ||
        t.includes("ai review") ||
        t.includes("ai evidence")
      ) {
        card.classList.add("nf-section-card", "nf-ai-card");
        matched = true;
      } else if (t.includes("final action") || t.includes("final decision")) {
        card.classList.add("nf-section-card", "nf-final-card");
        matched = true;
      } else if (
        t.includes("evidence") ||
        t.includes("mri") ||
        t.includes("longitudinal") ||
        t.includes("review complete")
      ) {
        card.classList.add("nf-section-card", "nf-evidence-card");
        matched = true;
      }

      if (matched && !card.querySelector(":scope > .nf-card-eyebrow")) {
        var eyebrow = document.createElement("div");
        eyebrow.className = "nf-card-eyebrow";
        eyebrow.setAttribute("aria-hidden", "true");

        if (card.classList.contains("nf-ai-card")) eyebrow.textContent = "AI evidence";
        else if (card.classList.contains("nf-passport-card")) eyebrow.textContent = "Evidence provenance";
        else if (card.classList.contains("nf-judgment-card")) eyebrow.textContent = "Reviewer judgment";
        else if (card.classList.contains("nf-final-card")) eyebrow.textContent = "Final disposition";
        else eyebrow.textContent = "Case evidence";

        card.insertBefore(eyebrow, card.firstChild);
      }
    });
  }

  function decorateButtons() {
    document.querySelectorAll("button,input[type='button'],input[type='submit']").forEach(function (b) {
      var t = ((b.value || "") + " " + textOf(b)).toLowerCase();
      if (
        t.includes("back") ||
        t.includes("previous") ||
        t.includes("download") ||
        t.includes("export")
      ) {
        b.classList.add("nf-secondary-button");
      }
    });
  }

  function updateProgress() {
    var bodyText = textOf(document.body);

    var patterns = [
      /case\s+(\d+)\s+of\s+(\d+)/i,
      /case\s+(\d+)\s*\/\s*(\d+)/i,
      /(\d+)\s+of\s+(\d+)\s+cases?/i
    ];

    var match = null;
    for (var i = 0; i < patterns.length; i++) {
      match = bodyText.match(patterns[i]);
      if (match) break;
    }

    var fill = document.getElementById("nf-progress-fill");
    var label = document.getElementById("nf-progress-label");
    if (!fill || !label) return;

    if (match) {
      var current = Math.max(1, parseInt(match[1], 10));
      var total = Math.max(current, parseInt(match[2], 10));
      var pct = Math.min(100, Math.max(0, (current / total) * 100));
      fill.style.width = pct + "%";
      label.textContent = "Case " + current + " of " + total;
    } else if (/review complete/i.test(bodyText)) {
      fill.style.width = "100%";
      label.textContent = "Complete";
    } else {
      fill.style.width = "0%";
      label.textContent = "12-case review";
    }
  }

  function accessibilityPass() {
    document.querySelectorAll("button").forEach(function (b) {
      if (!b.getAttribute("type")) b.setAttribute("type", "button");
    });

    document.querySelectorAll("textarea").forEach(function (el) {
      if (!el.getAttribute("spellcheck")) el.setAttribute("spellcheck", "true");
    });
  }

  function applyVisualEnhancements() {
    decorateSections();
    decorateButtons();
    updateProgress();
    accessibilityPass();
  }

  var queued = false;
  function scheduleApply() {
    if (queued) return;
    queued = true;
    window.requestAnimationFrame(function () {
      queued = false;
      applyVisualEnhancements();
    });
  }

  if (document.readyState === "loading") {
    document.addEventListener("DOMContentLoaded", applyVisualEnhancements, { once: true });
  } else {
    applyVisualEnhancements();
  }

  var observer = new MutationObserver(scheduleApply);
  observer.observe(document.documentElement, {
    childList: true,
    subtree: true,
    characterData: true
  });
})();
</script>
<!-- NF_UI_FACELIFT_SCRIPT_END -->
"""

print("✅ Professional visual layer defined.")

✅ Professional visual layer defined.


In [6]:
# Cell 6 — Inject visual layer WITHOUT rewriting existing application code

def inject_before_closing_tag(html: str, tag: str, block: str) -> str:
    pattern = re.compile(rf"</{tag}\s*>", re.I)
    matches = list(pattern.finditer(html))
    assert matches, f"Could not find closing </{tag}>."
    m = matches[-1]
    return html[:m.start()] + "\n" + block.strip() + "\n" + html[m.start():]

def inject_after_body_open(html: str, block: str) -> str:
    m = re.search(r"<body\b[^>]*>", html, flags=re.I)
    assert m, "Could not find <body>."
    return html[:m.end()] + "\n" + block.strip() + "\n" + html[m.end():]

# If this notebook is rerun on an already styled app, begin from the stable backup.
base_html = backup_index.read_text(encoding="utf-8")

candidate_html = base_html
candidate_html = inject_before_closing_tag(candidate_html, "head", FACELIFT_STYLE)
candidate_html = inject_after_body_open(candidate_html, FACELIFT_HEADER)
candidate_html = inject_before_closing_tag(candidate_html, "body", FACELIFT_FOOTER)
candidate_html = inject_before_closing_tag(candidate_html, "body", FACELIFT_SCRIPT)

assert "NF_UI_FACELIFT_STYLE_BEGIN" in candidate_html
assert "NF_UI_FACELIFT_HEADER_BEGIN" in candidate_html
assert "NF_UI_FACELIFT_FOOTER_BEGIN" in candidate_html
assert "NF_UI_FACELIFT_SCRIPT_BEGIN" in candidate_html

print("✅ Candidate HTML created.")
print("Candidate bytes:", len(candidate_html.encode("utf-8")))

✅ Candidate HTML created.
Candidate bytes: 32554


In [7]:
# Cell 7 — Copy untouched app assets into a candidate directory
if CANDIDATE_DIR.exists():
    shutil.rmtree(CANDIDATE_DIR)

shutil.copytree(BACKUP_DIR, CANDIDATE_DIR)
candidate_index = CANDIDATE_DIR / "index.html"
candidate_index.write_text(candidate_html, encoding="utf-8")

assert candidate_index.exists()
print("✅ UI candidate written:", candidate_index)

✅ UI candidate written: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app_ui_candidate/index.html


In [8]:
# Cell 8 — HARD scientific-integrity gate
#
# Strip ONLY the injected facelift blocks and require the remainder to equal
# the original pre-facelift HTML byte-for-byte.

BLOCK_PATTERNS = [
    r"\s*<!-- NF_UI_FACELIFT_STYLE_BEGIN -->.*?<!-- NF_UI_FACELIFT_STYLE_END -->\s*",
    r"\s*<!-- NF_UI_FACELIFT_HEADER_BEGIN -->.*?<!-- NF_UI_FACELIFT_HEADER_END -->\s*",
    r"\s*<!-- NF_UI_FACELIFT_FOOTER_BEGIN -->.*?<!-- NF_UI_FACELIFT_FOOTER_END -->\s*",
    r"\s*<!-- NF_UI_FACELIFT_SCRIPT_BEGIN -->.*?<!-- NF_UI_FACELIFT_SCRIPT_END -->\s*",
]

def strip_facelift(html: str) -> str:
    out = html
    for pat in BLOCK_PATTERNS:
        out = re.sub(pat, "\n", out, flags=re.I | re.S)
    # Injection adds surrounding newlines. Normalize ONLY those insertion seams
    # by comparing tokenized original structure around head/body markers.
    return out

stripped = strip_facelift(candidate_html)

# Stronger structural comparison:
# all original script blocks must remain byte-identical and in the same order.
scripts_after = SCRIPT_RE.findall(candidate_html)
# One new facelift script is expected, so remove it by marker.
scripts_after_original_only = [
    s for s in scripts_after
    if "nf-wish-professional-ui-script" not in s
]
script_hashes_after = [
    sha256_bytes(x.encode("utf-8")) for x in scripts_after_original_only
]

assert script_hashes_after == script_hashes_before, (
    "❌ Existing application script content changed. STOP."
)

# Every pre-existing critical marker must remain.
for marker in present_markers:
    assert marker.lower() in candidate_html.lower(), f"Critical marker missing: {marker}"

# Existing files other than index.html must remain byte-for-byte identical.
for old_file in sorted(BACKUP_DIR.rglob("*")):
    if not old_file.is_file():
        continue
    rel = old_file.relative_to(BACKUP_DIR)
    if rel.as_posix() == "index.html":
        continue
    new_file = CANDIDATE_DIR / rel
    assert new_file.exists(), f"Candidate missing existing asset: {rel}"
    assert sha256_file(old_file) == sha256_file(new_file), f"Existing asset changed: {rel}"

print("✅ EXISTING SCRIPT HASHES UNCHANGED")
print("✅ STUDY-CRITICAL MARKERS PRESERVED")
print("✅ ALL NON-INDEX ASSETS BYTE-FOR-BYTE UNCHANGED")
print("✅ SCIENTIFIC-INTEGRITY GATE: PASS")

✅ EXISTING SCRIPT HASHES UNCHANGED
✅ STUDY-CRITICAL MARKERS PRESERVED
✅ ALL NON-INDEX ASSETS BYTE-FOR-BYTE UNCHANGED
✅ SCIENTIFIC-INTEGRITY GATE: PASS


In [9]:
# Cell 9 — Researcher-only leakage gate
FORBIDDEN_TEXT_MARKERS = [
    "source_case_id",
    "ai_correctness",
    "reference_workflow_disposition_path_b",
    "path_a_reference_status",
    "final_researcher_scenario_key",
    "researcher_only",
    "researcher trajectory",
    "reference_volume",
    "reference volume",
]

FORBIDDEN_FILENAME_MARKERS = [
    "researcher",
    "answer_key",
    "scenario_key",
    "reference_workflow",
]

TEXT_SUFFIXES = {
    ".html", ".htm", ".js", ".mjs", ".cjs", ".json",
    ".css", ".txt", ".csv", ".md", ".xml"
}

leaks = []

for p in CANDIDATE_DIR.rglob("*"):
    if not p.is_file():
        continue

    rel = p.relative_to(CANDIDATE_DIR).as_posix().lower()

    for marker in FORBIDDEN_FILENAME_MARKERS:
        if marker in rel:
            leaks.append(f"filename:{rel} -> {marker}")

    if p.suffix.lower() in TEXT_SUFFIXES:
        txt = p.read_text(encoding="utf-8", errors="ignore").lower()
        for marker in FORBIDDEN_TEXT_MARKERS:
            if marker in txt:
                leaks.append(f"content:{rel} -> {marker}")

assert not leaks, (
    "❌ RESEARCHER-ONLY LEAKAGE DETECTED:\n" + "\n".join(leaks[:50])
)

print("✅ Researcher-only leakage gate: PASS")

✅ Researcher-only leakage gate: PASS


In [10]:
# Cell 10 — Accessibility / neutrality assertions on the injected design
lower = candidate_html.lower()

# No external visual dependencies added.
for bad in [
    "fonts.googleapis.com",
    "fonts.gstatic.com",
    "fontawesome",
    "cdnjs.cloudflare.com",
    "unpkg.com",
]:
    assert bad not in lower, f"External UI dependency detected: {bad}"

# No condition label is intentionally surfaced in our added header.
header_lower = FACELIFT_HEADER.lower()
for bias_marker in ["evidence-first", "ai-first", "sequence a", "sequence b"]:
    assert bias_marker not in header_lower, (
        f"Experimental condition leaked in visual shell: {bias_marker}"
    )

# Added UI makes no diagnostic/clinical-use claim.
assert "not for patient care or diagnostic use" in lower

print("✅ No external UI dependencies")
print("✅ No Evidence-First / AI-First condition leakage in header")
print("✅ Research-use disclaimer present")
print("✅ Interface-neutrality gate: PASS")

✅ No external UI dependencies
✅ No Evidence-First / AI-First condition leakage in header
✅ Research-use disclaimer present
✅ Interface-neutrality gate: PASS


In [11]:
# Cell 11 — Promote polished candidate to the frozen participant app
#
# The backup remains untouched so the original can always be restored.

if PROMOTE_TO_FROZEN_APP:
    # Replace index.html only. All original assets are already verified unchanged.
    shutil.copy2(candidate_index, INDEX)

    assert INDEX.read_text(encoding="utf-8") == candidate_html
    assert (BACKUP_DIR / "index.html").exists()

    print("✅ Professional UI promoted to frozen participant app.")
    print("Updated:", INDEX)
    print("Original backup:", BACKUP_DIR)
else:
    print("ℹ️ PROMOTE_TO_FROZEN_APP=False")
    print("Candidate retained at:", CANDIDATE_DIR)

✅ Professional UI promoted to frozen participant app.
Updated: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app/index.html
Original backup: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app_pre_ui_facelift


In [12]:
# Cell 12 — Generate a facelift audit manifest
audit = {
    "artifact": "NeuroFHIR-Review professional UI facelift",
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "frozen_app": str(FROZEN_APP),
    "backup_app": str(BACKUP_DIR),
    "candidate_app": str(CANDIDATE_DIR),
    "promoted": bool(PROMOTE_TO_FROZEN_APP),
    "pre_facelift_index_sha256": original_index_hash,
    "post_facelift_index_sha256": sha256_file(INDEX if PROMOTE_TO_FROZEN_APP else candidate_index),
    "existing_script_blocks": len(scripts_before),
    "existing_script_hashes_preserved": True,
    "non_index_assets_preserved": True,
    "researcher_only_leakage": False,
    "condition_assignment_exposed_by_header": False,
    "external_ui_dependencies_added": False,
    "design_principles": [
        "professional clinical-research appearance",
        "neutral AI/evidence visual weight",
        "responsive layout",
        "accessible focus states",
        "no study-logic modification",
    ],
}

audit_path = WISH_ROOT / "final_wish_pilot" / "ui_facelift_audit.json"
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")

print("✅ Audit manifest:", audit_path)

✅ Audit manifest: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/ui_facelift_audit.json


## What to do immediately after this notebook passes

Do **not** recruit anyone yet.

1. Rerun **Notebook 15A** to redeploy the updated frozen participant app to `/wish-review/`.
2. Open the public URL yourself on desktop and mobile.
3. Run corrected **Notebook 15B** for live deployment QA.
4. Only after the live UI looks right and 15B passes should the link go to Teri or another reviewer.

The visual facelift may be changed again **before real participant collection begins**. Once P003 starts, freeze the presentation and do not make design changes during the study.

In [13]:
# Cell 13 — Final gate
final_ok = (
    (BACKUP_DIR / "index.html").exists()
    and candidate_index.exists()
    and not leaks
    and script_hashes_after == script_hashes_before
)

if PROMOTE_TO_FROZEN_APP:
    final_ok = final_ok and ("NF_UI_FACELIFT_STYLE_BEGIN" in INDEX.read_text(encoding="utf-8"))

print("=" * 78)
print("NOTEBOOK 15A1 — NEUROFHIR-REVIEW PROFESSIONAL UI FACELIFT")
print("=" * 78)
print("Pre-facelift backup:                    PASS")
print("Existing study scripts unchanged:       PASS")
print("Non-index assets unchanged:             PASS")
print("Study-critical markers preserved:       PASS")
print("Researcher-only leakage:                PASS")
print("Condition label leakage:                PASS")
print("No external UI dependencies:            PASS")
print("Responsive professional UI injected:    PASS")
print("Promoted to frozen participant app:    ", "YES" if PROMOTE_TO_FROZEN_APP else "NO")
print("=" * 78)

assert final_ok

print("✅ NOTEBOOK 15A1 PROFESSIONAL UI FACELIFT GATE: TRUE")
print()
print("NEXT: rerun Notebook 15A to redeploy /wish-review/, then run Notebook 15B.")

NOTEBOOK 15A1 — NEUROFHIR-REVIEW PROFESSIONAL UI FACELIFT
Pre-facelift backup:                    PASS
Existing study scripts unchanged:       PASS
Non-index assets unchanged:             PASS
Study-critical markers preserved:       PASS
Researcher-only leakage:                PASS
Condition label leakage:                PASS
No external UI dependencies:            PASS
Responsive professional UI injected:    PASS
Promoted to frozen participant app:     YES
✅ NOTEBOOK 15A1 PROFESSIONAL UI FACELIFT GATE: TRUE

NEXT: rerun Notebook 15A to redeploy /wish-review/, then run Notebook 15B.
